In [1]:
pip install beautifulsoup4 requests lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
import json
import time
import random

# expanded list to reach that 200+ goal
SOURCES = {
    "moneycontrol": "https://www.moneycontrol.com/news/world/",
    "foxnews": "https://www.foxnews.com/world",
    "aljazeera": "https://www.aljazeera.com/where/middle-east/",
    "reuters": "https://www.reuters.com/world/",
    "the_hindu": "https://www.thehindu.com/news/international/",
    "defense_news": "https://www.defensenews.com/global/",
    "ndtv": "https://www.ndtv.com/world-news",
    "times_of_india": "https://timesofindia.indiatimes.com/world",
    "the_guardian": "https://www.theguardian.com/world",
    "france24": "https://www.france24.com/en/world/",
    "cnn_world": "https://edition.cnn.com/world",
    "ap_news": "https://apnews.com/hub/world-news",
    "dw_news": "https://www.dw.com/en/world/s-33",
    "telegraph": "https://www.telegraph.co.uk/world-news/"
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
}

def get_article_content(url):
    try:
        # Short timeout to keep the script moving
        res = requests.get(url, headers=HEADERS, timeout=8)
        if res.status_code != 200: return None
        
        soup = BeautifulSoup(res.text, "lxml")
        # Grabbing common article text tags
        paragraphs = soup.find_all("p")
        # Filter: Only keep paragraphs that look like real sentences
        text = " ".join([p.get_text().strip() for p in paragraphs if len(p.get_text()) > 60])
        
        return text[:2000] if len(text) > 200 else None
    except:
        return None

all_data = []
target_count = 200

print(f"Starting crawl to reach {target_count} articles...")

for name, start_url in SOURCES.items():
    if len(all_data) >= target_count:
        break

    print(f"\n--- Scouting {name.upper()} ---")
    try:
        response = requests.get(start_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(response.text, "lxml")
        
        # Collect unique links
        links = []
        for a in soup.find_all("a", href=True):
            link = a['href']
            if link.startswith("/"):
                # Construct absolute URL
                base = start_url.split(".com")[0] + ".com" if ".com" in start_url else start_url.split(".in")[0] + ".in"
                link = base + link
            
            # Filter for links that look like articles (usually have dates or long slugs)
            if any(key in link for key in ["/2026/", "/2025/", "/articles/", "/news/"]) and link not in links:
                if "facebook.com" not in link and "twitter.com" not in link:
                    links.append(link)

        print(f"Found {len(links)} potential stories. Starting deep scrape...")

        for link in links:
            if len(all_data) >= target_count:
                break
                
            content = get_article_content(link)
            if content:
                all_data.append({
                    "source": name,
                    "url": link,
                    "content": content
                })
                print(f"  [{len(all_data)}] Saved: {link[:60]}...")
                
                # Randomized delay to mimic human browsing and avoid IP bans
                time.sleep(random.uniform(0.5, 1.5)) 
            
    except Exception as e:
        print(f"Error skipping {name}: {e}")

# Final Save
with open("large_news_dataset.json", "w") as f:
    json.dump(all_data, f, indent=4)

print(f"\nMission accomplished! Total articles: {len(all_data)}")

Starting crawl to reach 200 articles...

--- Scouting MONEYCONTROL ---
Found 53 potential stories. Starting deep scrape...
  [1] Saved: https://www.moneycontrol.com/news/...
  [2] Saved: https://www.moneycontrol.com/news/business/...
  [3] Saved: https://www.moneycontrol.com/news/business/economy/...
  [4] Saved: https://www.moneycontrol.com/news/business/companies/...
  [5] Saved: https://www.moneycontrol.com/news/business/mutual-funds/...
  [6] Saved: https://www.moneycontrol.com/news/business/personal-finance/...
  [7] Saved: https://www.moneycontrol.com/news/business/ipo/...
  [8] Saved: https://www.moneycontrol.com/news/business/startup/...
  [9] Saved: https://www.moneycontrol.com/news/business/real-estate/...
  [10] Saved: https://www.moneycontrol.com/news/india/...
  [11] Saved: https://www.moneycontrol.com/news/politics/...
  [12] Saved: https://www.moneycontrol.com/news/business/markets/...
  [13] Saved: https://www.moneycontrol.com/news/business/stocks/...
  [14] Saved: http

In [5]:
pip install newspaper3k lxml[html_clean]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
#VERSION 1

import json
import time
import random
import requests
from bs4 import BeautifulSoup
from newspaper import Article, Config

# 1. Expanded Source List to ensure we hit 200+ articles
SOURCES = {
    "moneycontrol_world": "https://www.moneycontrol.com/news/world/",
    "aljazeera": "https://www.aljazeera.com/where/middle-east/",
    "defense_news": "https://www.defensenews.com/global/",
    "reuters_world": "https://www.reuters.com/world/",
    "fox_world": "https://www.foxnews.com/world",
    "the_hindu_intl": "https://www.thehindu.com/news/international/",
    "dw_world": "https://www.dw.com/en/world/s-33",
    "france24": "https://www.france24.com/en/tag/war/",
    "telegraph_world": "https://www.telegraph.co.uk/world-news/",
    "times_of_india": "https://timesofindia.indiatimes.com/world",
    "ap_news": "https://apnews.com/hub/world-news",
    "defense_one": "https://www.defenseone.com/threats/"
}

# 2. Targeted US-Iran & Global War Keywords
WAR_KEYWORDS = [
    # General Conflict
    "war", "conflict", "missile", "attack", "strike", "military", "army", 
    "defense", "bombing", "airstrike", "ceasefire", "casualty", "invasion",
    
    # US-Iran Specific
    "iran", "tehran", "pentagon", "centcom", "irgc", "quds force", "white house",
    "strait of hormuz", "persian gulf", "red sea", "gulf of oman", "bab el-mandeb",
    "nuclear", "enrichment", "sanctions", "oil tanker", "drone swarm", 
    "ballistic", "carrier strike group", "b-52", "cyberattack",
    
    # Regional Proxies & Allies
    "hezbollah", "houthis", "militia", "proxy", "israel", "tel aviv", 
    "syria", "iraq", "lebanon", "axis of resistance",
    
    # Escalation
    "retaliation", "escalation", "ultimatum", "deterrence", "blockade", 
    "assassination", "red line", "ukraine", "russia", "gaza"
]

# 3. Scraper Configuration
config = Config()
config.browser_user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
config.request_timeout = 15

def is_war_related(text, title):
    """Returns True if the article matches our war-specific keyword list."""
    combined = (str(text) + " " + str(title)).lower()
    # Check if at least TWO keywords appear to ensure higher relevance
    matches = [word for word in WAR_KEYWORDS if word in combined]
    return len(matches) >= 1  # Change to >= 2 for even stricter filtering

all_articles = []
target_count = 200

print(f"🚀 Initializing War Scraper. Target: {target_count} articles.")

for name, start_url in SOURCES.items():
    if len(all_articles) >= target_count:
        break

    print(f"\n--- Checking {name.upper()} ---")
    try:
        # Use requests to get the listing page
        res = requests.get(start_url, headers={'User-Agent': config.browser_user_agent}, timeout=15)
        soup = BeautifulSoup(res.text, "lxml")
        
        # Extract links from the page
        links = []
        for a in soup.find_all("a", href=True):
            link = a['href']
            if link.startswith("/"):
                # Reconstruct relative links
                domain = start_url.split("//")[1].split("/")[0]
                link = f"https://{domain}{link}"
            
            # Basic validation to ensure it looks like a news story link
            if any(path in link for path in ["/news/", "/world/", "/2026/", "/2025/", "/articles/"]):
                if link not in links and "facebook" not in link and "twitter" not in link:
                    links.append(link)

        print(f"Found {len(links)} links. Filtering content...")

        # Use newspaper3k for deep cleaning
        for link in links:
            if len(all_articles) >= target_count:
                break
            
            try:
                article = Article(link, config=config)
                article.download()
                article.parse()
                
                if is_war_related(article.text, article.title):
                    # Success! Clean content found.
                    all_articles.append({
                        "id": len(all_articles) + 1,
                        "source": name,
                        "title": article.title,
                        "url": link,
                        "date": str(article.publish_date) if article.publish_date else "N/A",
                        "content": article.text[:3000],  # Ad-free, clean text body
                        "top_image": article.top_image
                    })
                    print(f"  [{len(all_articles)}] Captured: {article.title[:55]}...")
                    
                    # Prevent IP rate-limiting
                    time.sleep(random.uniform(0.7, 1.5))
            except Exception:
                continue

    except Exception as e:
        print(f"Skipping {name} due to error: {e}")

# 4. Save to JSON
output_file = "war_dataset_2026.json"
with open(output_file, "w", encoding='utf-8') as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

print(f"\n✅ SUCCESS: {len(all_articles)} articles saved to {output_file}")

🚀 Initializing War Scraper. Target: 200 articles.

--- Checking MONEYCONTROL_WORLD ---
Found 122 links. Filtering content...
  [1] Captured: Business News, Economic News, Indian Stock Market News...
  [2] Captured: Economy News - Latest News on Indian Economy, Governmen...
  [3] Captured: Company Business News: Latest Indian Companies News, Co...
  [4] Captured: IPO News- Latest IPO News Today | Upcoming IPO |IPO Iss...
  [5] Captured: Business News, Economic News, Indian Stock Market News...
  [6] Captured: Real Estate News: Latest Residential & Commercial Prope...
  [7] Captured: India News: Latest India Live & Breaking News & Headlin...
  [8] Captured: World News: Latest International News & Top World Headl...
  [9] Captured: Current Political News, Political News, Politics News, ...
  [10] Captured: Share Market News Today: Indian Stock Market News and L...
  [11] Captured: Stock Market Today | Bombay Stock Exchange Updates | Gl...
  [12] Captured: Business News, Economic News, Ind

In [1]:
#VERSION 2

import json
import time
import random
import requests
import os
from bs4 import BeautifulSoup
from newspaper import Article, Config

# Create an images folder if it doesn't exist
IMAGE_FOLDER = "downloaded_images"
if not os.path.exists(IMAGE_FOLDER):
    os.makedirs(IMAGE_FOLDER)

SOURCES = {
    "moneycontrol": "https://www.moneycontrol.com/news/world/",
    "aljazeera": "https://www.aljazeera.com/where/middle-east/",
    "defense_news": "https://www.defensenews.com/global/",
    "reuters": "https://www.reuters.com/world/",
    "fox": "https://www.foxnews.com/world",
    "the_hindu": "https://www.thehindu.com/news/international/",
    "dw": "https://www.dw.com/en/world/s-33",
    "france24": "https://www.france24.com/en/world/",
    "telegraph": "https://www.telegraph.co.uk/world-news/",
    "times_of_india": "https://timesofindia.indiatimes.com/world",
    "ap_news": "https://apnews.com/hub/world-news",
    "defense_one": "https://www.defenseone.com/threats/",
    "cnn_world": "https://edition.cnn.com/world",
    "bbc_world": "https://www.bbc.com/news/world"
}

WAR_KEYWORDS = [
    "war", "conflict", "missile", "attack", "strike", "military", "army", 
    "defense", "bombing", "airstrike", "ceasefire", "casualty", "invasion",
    "iran", "tehran", "irgc", "centcom", "strait of hormuz", "nuclear", 
    "drone", "ballistic", "carrier strike group", "hezbollah", "houthi", 
    "israel", "syria", "iraq", "ukraine", "russia", "gaza", "retaliation"
]

config = Config()
config.browser_user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
config.request_timeout = 15

def download_image(url, article_id):
    """Downloads the image and returns the local file path."""
    if not url:
        return None
    try:
        img_res = requests.get(url, headers={'User-Agent': config.browser_user_agent}, timeout=10)
        if img_res.status_code == 200:
            file_extension = url.split(".")[-1].split("?")[0] # handle urls with params
            if len(file_extension) > 4: file_extension = "jpg" # fallback
            
            filename = f"article_{article_id}.{file_extension}"
            filepath = os.path.join(IMAGE_FOLDER, filename)
            
            with open(filepath, 'wb') as f:
                f.write(img_res.content)
            return filepath
    except Exception as e:
        print(f"      [Image Error]: {e}")
    return None

def is_war_related(text, title):
    combined = (str(text) + " " + str(title)).lower()
    matches = [word for word in WAR_KEYWORDS if word in combined]
    return len(matches) >= 1

all_articles = []
target_count = 800  # Set to 800 as requested

print(f"🚀 Initializing War Scraper. Target: {target_count} articles.")

for name, start_url in SOURCES.items():
    if len(all_articles) >= target_count:
        break

    print(f"\n--- Checking {name.upper()} ---")
    try:
        res = requests.get(start_url, headers={'User-Agent': config.browser_user_agent}, timeout=15)
        soup = BeautifulSoup(res.text, "lxml")
        
        links = []
        for a in soup.find_all("a", href=True):
            link = a['href']
            if link.startswith("/"):
                domain = start_url.split("//")[1].split("/")[0]
                link = f"https://{domain}{link}"
            
            if any(path in link for path in ["/news/", "/world/", "/2026/", "/2025/", "/articles/"]):
                if link not in links and "facebook" not in link and "twitter" not in link:
                    links.append(link)

        print(f"Found {len(links)} links. Scraping...")

        for link in links:
            if len(all_articles) >= target_count:
                break
            
            try:
                article = Article(link, config=config)
                article.download()
                article.parse()
                
                if is_war_related(article.text, article.title) and len(article.text) > 400:
                    article_id = len(all_articles) + 1
                    
                    # DOWNLOAD THE IMAGE LOCALLY
                    local_img_path = download_image(article.top_image, article_id)
                    
                    all_articles.append({
                        "id": article_id,
                        "source": name,
                        "title": article.title,
                        "url": link,
                        "date": str(article.publish_date) if article.publish_date else "N/A",
                        "content": article.text[:3000],
                        "remote_image_url": article.top_image,
                        "local_image_path": local_img_path
                    })
                    print(f"  [{len(all_articles)}] Captured: {article.title[:45]}...")
                    time.sleep(random.uniform(0.5, 1.2))
            except Exception:
                continue

    except Exception as e:
        print(f"Skipping {name}: {e}")

# Save JSON
output_file = "war_dataset_large.json"
with open(output_file, "w", encoding='utf-8') as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

print(f"\n✅ SUCCESS: {len(all_articles)} articles and images saved.")

🚀 Initializing War Scraper. Target: 800 articles.

--- Checking MONEYCONTROL ---
Found 122 links. Scraping...
  [1] Captured: Business News, Economic News, Indian Stock Ma...
  [2] Captured: Economy News - Latest News on Indian Economy,...
  [3] Captured: Company Business News: Latest Indian Companie...
  [4] Captured: IPO News- Latest IPO News Today | Upcoming IP...
  [5] Captured: Business News, Economic News, Indian Stock Ma...
  [6] Captured: Real Estate News: Latest Residential & Commer...
  [7] Captured: India News: Latest India Live & Breaking News...
  [8] Captured: World News: Latest International News & Top W...
  [9] Captured: Current Political News, Political News, Polit...
  [10] Captured: Share Market News Today: Indian Stock Market ...
  [11] Captured: Stock Market Today | Bombay Stock Exchange Up...
  [12] Captured: Business News, Economic News, Indian Stock Ma...
  [13] Captured: Business News, Economic News, Indian Stock Ma...
  [14] Captured: Trending News on Busines

In [2]:
#VERSION 3

import json
import time
import random
import requests
import os
from bs4 import BeautifulSoup
from newspaper import Article, Config

# --- CONFIG ---
IMAGE_FOLDER = "war_images"
if not os.path.exists(IMAGE_FOLDER):
    os.makedirs(IMAGE_FOLDER)

# List of realistic desktop User-Agents to rotate
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0"
]

def get_stealth_headers():
    """Generates a full browser-like header stack."""
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Cache-Control": "max-age=0",
        "Referer": "https://www.google.com/" # Pretend we came from Google
    }

# --- SOURCES ---
SOURCES = {
    "aljazeera_war": "https://www.aljazeera.com/tag/war-news/",
    "ap_war": "https://apnews.com/hub/war-and-unrest",
    "reuters_world": "https://www.reuters.com/world/",
    "scmp_intl": "https://www.scmp.com/news/world",
    "defense_news": "https://www.defensenews.com/global/",
    "telegraph_world": "https://www.telegraph.co.uk/world-news/",
    "dw_world": "https://www.dw.com/en/world/s-33",
    "moneycontrol": "https://www.moneycontrol.com/news/world/",
    "fox": "https://www.foxnews.com/world",
    "the_hindu": "https://www.thehindu.com/news/international/",
    "france24": "https://www.france24.com/en/world/",
    "times_of_india": "https://timesofindia.indiatimes.com/world",
    "defense_one": "https://www.defenseone.com/threats/",
    "cnn_world": "https://edition.cnn.com/world",
    "bbc_world": "https://www.bbc.com/news/world"
}

# --- MAIN LOGIC ---
all_articles = []
target_count = 800

print(f"🚀 Starting Stealth Scraper (Target: {target_count})")

for name, start_url in SOURCES.items():
    if len(all_articles) >= target_count: break
    print(f"\n--- Scraping {name.upper()} ---")
    
    # 1. GENERATE STEALTH HEADERS FOR THIS SITE
    current_headers = get_stealth_headers()
    
    try:
        # Use our new headers to get the link list
        res = requests.get(start_url, headers=current_headers, timeout=15)
        soup = BeautifulSoup(res.text, "lxml")
        
        links = []
        for a in soup.find_all("a", href=True):
            href = a['href']
            if href.startswith("/"):
                domain = start_url.split("//")[1].split("/")[0]
                href = f"https://{domain}{href}"
            
            # Refined link filter
            if any(p in href for p in ["/news/", "/world/", "/articles/", "/tag/"]) and href not in links:
                links.append(href)

        print(f"Found {len(links)} links. Scraping articles...")

        for link in links:
            if len(all_articles) >= target_count: break
            try:
                # 2. APPLY STEALTH TO NEWSPAPER3K
                config = Config()
                config.headers = get_stealth_headers() # Rotate headers for every article request
                config.request_timeout = 15
                
                article = Article(link, config=config)
                article.download()
                article.parse()
                
                if len(article.text) > 400: # Removed keyword filter to boost volume
                    current_id = len(all_articles) + 1
                    all_articles.append({
                        "id": current_id,
                        "source": name,
                        "title": article.title,
                        "url": link,
                        "content": article.text[:4000],
                        "image_url": article.top_image
                    })
                    print(f"  [{current_id}] Captured: {article.title[:50]}...")
                    
                    # 3. BEHAVIORAL BYPASS: Variable Sleep
                    # Mimics a human reading for a few seconds
                    time.sleep(random.uniform(1.5, 3.5)) 
            except:
                continue
    except Exception as e:
        print(f"Blocked on {name}: {e}")

# Save JSON
with open("stealth_war_dataset.json", "w", encoding='utf-8') as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

🚀 Starting Stealth Scraper (Target: 800)

--- Scraping ALJAZEERA_WAR ---
Found 14 links. Scraping articles...
  [1] Captured: US pilot from downed F-15E plane rescued in Iran: ...
  [2] Captured: Marco Rubio says he has stripped Qassem Soleimani’...
  [3] Captured: Why an attack on Bushehr nuclear plant would be ca...
  [4] Captured: Oman, Iran discuss smooth transit in Strait of Hor...
  [5] Captured: Iran’s ex-FM Zarif proposes peace roadmap; Gulf po...
  [6] Captured: Iran war: What is happening on day 37 of US-Israel...
  [7] Captured: US satellite firm Planet Labs announces blackout o...
  [8] Captured: Kuwait’s power, water plants damaged as Iran keeps...

--- Scraping AP_WAR ---
Found 0 links. Scraping articles...

--- Scraping REUTERS_WORLD ---
Found 0 links. Scraping articles...

--- Scraping SCMP_INTL ---
Found 0 links. Scraping articles...

--- Scraping DEFENSE_NEWS ---
Found 8 links. Scraping articles...
  [9] Captured: US special forces rescue second F-15 airman from I...


In [8]:
#VERSION 4

import json
import time
import random
import requests
import os
import re
from bs4 import BeautifulSoup
from newspaper import Article, Config

# --- CONFIG ---
IMAGE_FOLDER = "war_images"
if not os.path.exists(IMAGE_FOLDER):
    os.makedirs(IMAGE_FOLDER)

# Simple, highly-compatible headers
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

# Targeted Keywords for US-Iran and War Scenario
WAR_KEYWORDS = [
    "war", "conflict", "missile", "attack", "strike", "military", "irgc", "centcom", 
    "strait of hormuz", "nuclear", "drone", "hezbollah", "houthi", "israel", "iran", 
    "russia", "ukraine", "gaza", "airstrike", "invasion", "trump", "soleimani"
]

def download_image(url, article_id):
    if not url or not url.startswith("http"): return None
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            ext = url.split('.')[-1].split('?')[0].lower()[:4]
            if ext not in ['jpg', 'jpeg', 'png', 'webp']: ext = 'jpg'
            filename = f"war_img_{article_id}.{ext}"
            filepath = os.path.join(IMAGE_FOLDER, filename)
            with open(filepath, 'wb') as f:
                f.write(res.content)
            return filename
    except: return None

def is_war_related(text, title):
    combined = (str(text) + " " + str(title)).lower()
    return any(word in combined for word in WAR_KEYWORDS)

# --- SOURCES ---
SOURCES = {
    "aljazeera_war": "https://www.aljazeera.com/tag/war-news/",
    "ap_war": "https://apnews.com/hub/war-and-unrest",
    "reuters_world": "https://www.reuters.com/world/",
    "scmp_intl": "https://www.scmp.com/news/world",
    "defense_news": "https://www.defensenews.com/global/",
    "telegraph_world": "https://www.telegraph.co.uk/world-news/",
    "dw_world": "https://www.dw.com/en/world/s-33",
    "moneycontrol": "https://www.moneycontrol.com/news/world/",
    "fox": "https://www.foxnews.com/world",
    "the_hindu": "https://www.thehindu.com/news/international/",
    "france24": "https://www.france24.com/en/world/",
    "times_of_india": "https://timesofindia.indiatimes.com/world",
    "defense_one": "https://www.defenseone.com/threats/",
    "cnn_world": "https://edition.cnn.com/world",
    "bbc_world": "https://www.bbc.com/news/world"
}

all_articles = []
target_count = 800

print(f"🚀 Version 4: Brute-Force Starting. Target: {target_count}")

for name, start_url in SOURCES.items():
    if len(all_articles) >= target_count: break
    print(f"\n--- Checking {name.upper()} ---")
    
    try:
        res = requests.get(start_url, headers=HEADERS, timeout=15)
        # If requests fails to find links, we try to use newspaper's built-in discovery
        soup = BeautifulSoup(res.text, "lxml")
        links = []
        
        # Method 1: BS4 Discovery
        for a in soup.find_all("a", href=True):
            href = a['href']
            if href.startswith("/"):
                domain = start_url.split("//")[1].split("/")[0]
                href = f"https://{domain}{href}"
            
            # Stricter link filter to avoid "UK Election" garbage
            path_match = any(p in href for p in ["/news/", "/world/", "/2026/", "/2025/"])
            topic_match = any(k in href.lower() for k in ["war", "iran", "missile", "conflict", "israel"])
            
            if path_match and topic_match and href not in links:
                links.append(href)

        print(f"Found {len(links)} targeted links.")

        for link in links:
            if len(all_articles) >= target_count: break
            try:
                config = Config()
                config.headers = HEADERS
                article = Article(link, config=config)
                article.download()
                article.parse()
                
                # Check Substance and Relevance
                if len(article.text) > 400 and is_war_related(article.text, article.title):
                    current_id = len(all_articles) + 1
                    
                    # RESTORED: Image Downloading
                    img_file = download_image(article.top_image, current_id)
                    
                    all_articles.append({
                        "id": current_id,
                        "source": name,
                        "title": article.title,
                        "url": link,
                        "date": str(article.publish_date) if article.publish_date else "N/A",
                        "content": article.text[:4000],
                        "top_image": img_file
                    })
                    print(f"  [{current_id}] Captured: {article.title[:50]}...")
                    time.sleep(random.uniform(0.5, 1.5))
            except: continue
            
    except Exception as e:
        print(f"Error on {name}: {e}")

# Save JSON
with open("war_dataset_v4.json", "w", encoding='utf-8') as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

print(f"\n✅ DONE. Total unique articles: {len(all_articles)}")

🚀 Version 4: Brute-Force Starting. Target: 800

--- Checking ALJAZEERA_WAR ---
Found 8 targeted links.
  [1] Captured: US pilot from downed F-15E plane rescued in Iran: ...
  [2] Captured: Oman, Iran discuss smooth transit in Strait of Hor...
  [3] Captured: Iran’s ex-FM Zarif proposes peace roadmap; Gulf po...
  [4] Captured: Iran war: What is happening on day 37 of US-Israel...
  [5] Captured: US satellite firm Planet Labs announces blackout o...
  [6] Captured: Trump threatens ‘hell’ for Iran over Hormuz Strait...

--- Checking AP_WAR ---
Found 0 targeted links.

--- Checking REUTERS_WORLD ---
Found 0 targeted links.

--- Checking SCMP_INTL ---
Found 29 targeted links.
  [7] Captured: Israeli strikes kill 11 in Lebanon, Israel announc...
  [8] Captured: Pope Leo calls for peace, dialogue amid Ukraine, I...
  [9] Captured: UAE, Bahrain and Kuwait lose water, energy infrast...
  [10] Captured: UAE, Bahrain and Kuwait lose water, energy infrast...
  [11] Captured: Ukraine’s Zelensky fe

In [7]:
#VERSION 5 - SOURCES directly go to search bar

import json
import time
import random
import requests
import os
import re
from bs4 import BeautifulSoup
from newspaper import Article, Config

# --- CONFIG ---
IMAGE_FOLDER = "war_images_two"
if not os.path.exists(IMAGE_FOLDER):
    os.makedirs(IMAGE_FOLDER)

# Simple, highly-compatible headers
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

# Targeted Keywords 
WAR_KEYWORDS = [
    "war", "conflict", "missile", "attack", "strike", "military", "irgc", "centcom",
    "strait of hormuz", "nuclear", "drone", "hezbollah", "houthi", "israel", "iran",
    "russia", "ukraine", "gaza", "airstrike", "invasion", "trump", "soleimani",
    "india", "pakistan", "loc", "line of control", "kashmir", "balakot", "pulwama",
    "surgical strike", "border clash", "idf", "iron dome", "hamas", "west bank",
    "jerusalem", "quds force", "ayatollah", "shia militia", "proxy war", "donbas",
    "crimea", "kyiv", "mariupol", "kharkiv", "nato", "sanctions", "mobilization",
    "frontline", "occupation", "ceasefire", "escalation", "casualties", "shelling",
    "artillery", "rebels", "terrorist attack", "peace talks", "blockade", "special forces",
    "tanks", "fighter jets", "ballistic missile", "hypersonic", "naval blockade",
    "air defense"
]

def download_image(url, article_id):
    if not url or not url.startswith("http"): return None
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            ext = url.split('.')[-1].split('?')[0].lower()[:4]
            if ext not in ['jpg', 'jpeg', 'png', 'webp']: ext = 'jpg'
            filename = f"war_img_{article_id}.{ext}"
            filepath = os.path.join(IMAGE_FOLDER, filename)
            with open(filepath, 'wb') as f:
                f.write(res.content)
            return filename
    except: return None

def is_war_related(text, title):
    combined = (str(text) + " " + str(title)).lower()
    return any(word in combined for word in WAR_KEYWORDS)

# --- SOURCES ---
SOURCES = {
    "aljazeera_tag": "https://www.aljazeera.com/tag/iran-israel-conflict/",
    "moneycontrol_news": "https://www.moneycontrol.com/news/tags/iran-israel-war.html",
    "scmp_iran": "https://www.scmp.com/topics/iran-israel-conflict",
    "defense_news_intl": "https://www.defensenews.com/global/asia-pacific/",
    "hindustan_times": "https://www.hindustantimes.com/world-news/iran-israel-war-news-live-updates-today",
    "strait_times": "https://www.straitstimes.com/tags/iran-israel-conflict",
    "arab_news": "https://www.arabnews.com/tags/iran-israel-conflict",
    "times_of_israel": "https://www.timesofisrael.com/topic/iran-israel-conflict/",
    "jerusalem_post": "https://www.jpost.com/middle-east/iran-news",
    "the_print": "https://theprint.in/tag/iran-israel-war/",
    "wion_news": "https://www.wionews.com/topic/iran-israel-conflict",
    "france24_search": "https://www.france24.com/en/tag/iran/",
    "bbc_world": "https://www.bbc.com/news/world",
    "the_hindu": "https://www.thehindu.com/news/"
}

all_articles = []
target_count = 800

print(f"🚀 Version 5. Target: {target_count}")

for name, start_url in SOURCES.items():
    if len(all_articles) >= target_count: break
    print(f"\n--- Checking {name.upper()} ---")
    
    try:
        res = requests.get(start_url, headers=HEADERS, timeout=15)
        # If requests fails to find links, we try to use newspaper's built-in discovery
        soup = BeautifulSoup(res.text, "lxml")
        links = []
        
        # Method 1: BS4 Discovery
        for a in soup.find_all("a", href=True):
            href = a['href']
            if href.startswith("/"):
                domain = start_url.split("//")[1].split("/")[0]
                href = f"https://{domain}{href}"
            
            # Stricter link filter to avoid "UK Election" garbage
            path_match = any(p in href for p in ["/news/", "/world/", "/2026/", "/2025/"])
            topic_match = any(k in href.lower() for k in ["war", "iran", "missile", "conflict", "israel"])
            
            if path_match and topic_match and href not in links:
                links.append(href)

        print(f"Found {len(links)} targeted links.")

        for link in links:
            if len(all_articles) >= target_count: break
            try:
                config = Config()
                config.headers = HEADERS
                article = Article(link, config=config)
                article.download()
                article.parse()
                
                # Check Substance and Relevance
                if len(article.text) > 400 and is_war_related(article.text, article.title):
                    current_id = len(all_articles) + 1
                    
                    # RESTORED: Image Downloading
                    img_file = download_image(article.top_image, current_id)
                    
                    all_articles.append({
                        "id": current_id,
                        "source": name,
                        "title": article.title,
                        "url": link,
                        "date": str(article.publish_date) if article.publish_date else "N/A",
                        "content": article.text[:4000],
                        "top_image": img_file
                    })
                    print(f"  [{current_id}] Captured: {article.title[:50]}...")
                    time.sleep(random.uniform(0.5, 1.5))
            except: continue
            
    except Exception as e:
        print(f"Error on {name}: {e}")

# Save JSON
with open("war_dataset_v5.json", "w", encoding='utf-8') as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

print(f"\n✅ DONE. Total unique articles: {len(all_articles)}")

🚀 Version 5. Target: 800

--- Checking ALJAZEERA_TAG ---
Found 8 targeted links.
  [1] Captured: US pilot from downed F-15E plane rescued in Iran: ...
  [2] Captured: Oman, Iran discuss smooth transit in Strait of Hor...
  [3] Captured: Iran’s ex-FM Zarif proposes peace roadmap; Gulf po...
  [4] Captured: Iran war: What is happening on day 37 of US-Israel...
  [5] Captured: US satellite firm Planet Labs announces blackout o...
  [6] Captured: Trump threatens ‘hell’ for Iran over Hormuz Strait...

--- Checking MONEYCONTROL_NEWS ---
Found 17 targeted links.
  [7] Captured: 'Unprecedented begging': Iran criticises Trump’s e...
  [8] Captured: Iran mocks Trump, calls US president ‘teenager’ ov...
  [9] Captured: 'Everything blown to pieces': Iran strike destroys...
  [10] Captured: ‘They are negotiating now’: Donald Trump sees 'goo...
  [11] Captured: Trump Says Iran Essentially Decimated, Hard Part I...
  [12] Captured: Why Gold & Silver Are Falling Despite US–Iran War...
  [13] Captured: